In [12]:
# Random forest steps
# 1. draw K bootstrap samples from the training data
# 2. for each bootstrap sample, build a decision tree model: do not consider all variables for splitting
 # a. consider randomly selected p-variables for splitting at each node (p<t total number of variables)
 # b. use the variable with highest information gain for splitting out of the p-variables
 # c. go to each child node and repeat
 # d. grow each tree to the largest extent possible (no pruning)
# 3. collate the results for the new data points based on the class with majority votes from all K trees

# Hyperparameters: p =Sqrt(t) for classification, should be low as possible, start with sqrt(t) and tune from there
# K = number of trees, should be high as possible, start with 50 and 500 and tune from there 
# max_depth, if too less then underfitting, if too high then overfitting, tune from there

# CASE Study: Car Accidents Prediction. Predict car accidents based on 22 sensor variables

import pandas as pd
car_train = pd.read_csv('car_sensors.csv')

car_train.head()

car_train.info()

car_train.describe()






<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33239 entries, 0 to 33238
Data columns (total 23 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   safe    33239 non-null  int64  
 1   S1      33239 non-null  float64
 2   S2      33239 non-null  float64
 3   S3      33239 non-null  float64
 4   S4      33239 non-null  int64  
 5   S5      33239 non-null  float64
 6   S6      33239 non-null  float64
 7   S7      33239 non-null  float64
 8   S8      33239 non-null  int64  
 9   S9      33239 non-null  int64  
 10  S10     33239 non-null  float64
 11  S11     33239 non-null  int64  
 12  S12     33239 non-null  int64  
 13  S13     33239 non-null  int64  
 14  S14     33239 non-null  int64  
 15  S15     33239 non-null  float64
 16  S16     33239 non-null  float64
 17  S17     33239 non-null  int64  
 18  S18     33239 non-null  float64
 19  S19     33239 non-null  int64  
 20  S20     33239 non-null  float64
 21  S21     33239 non-null  int64  
 22

,safe,S1,S2,S3,S4,S5,S6,S7,S8,S9,...,S13,S14,S15,S16,S17,S18,S19,S20,S21,S22
count,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,...,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000,33239.000000
mean,0.575800,35.460193,12.036078,0.175710,837.136617,77.981648,10.437655,103.317494,0.279190,-4.048918,...,0.878606,63.347062,1.392497,-0.040675,572.367370,20.008582,0.179037,12.620019,3.305304,11.598367
std,0.494228,7.265388,3.749871,0.330817,2187.282185,18.947660,13.960038,127.527328,0.987574,35.902237,...,0.326589,18.937485,5.401432,0.403748,297.919127,63.433278,0.383389,11.552932,1.249511,8.980693
min,0.000000,-22.159000,-45.614600,0.038920,136.000000,0.262224,0.000000,0.000000,0.000000,-250.000000,...,0.000000,0.000000,0.000000,-4.305000,240.000000,0.000000,0.000000,0.000000,1.000000,1.679530
25%,0.000000,31.793350,9.915270,0.092110,668.000000,66.666700,0.000000,0.000000,0.000000,-8.000000,...,1.000000,52.000000,0.000000,-0.175000,255.000000,1.487500,0.000000,0.000000,3.000000,7.965390
50%,1.000000,34.156300,11.434900,0.105083,800.000000,75.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,67.000000,0.000000,0.000000,511.000000,3.018750,0.000000,12.800000,4.000000,10.770500
75%,1.000000,37.366450,13.721450,0.137516,900.000000,89.820400,28.177000,213.517000,0.000000,6.000000,...,1.000000,73.000000,0.000000,0.070000,767.000000,7.481250,0.000000,21.900000,4.000000,15.245350
max,1.000000,101.341000,71.154000,11.720000,228812.000000,441.176000,96.839000,359.958000,4.000000,250.000000,...,1.000000,126.000000,52.000000,3.605000,1023.000000,481.512000,1.000000,71.500000,7.000000,262.447000


In [13]:
car_train ['safe'].value_counts()

safe
1    19139
0    14100
Name: count, dtype: int64

In [14]:
# Model Building and Validation

# Build a decision tree model first, and then build random forest on top of it, compare results. 

# the later one should have better accuracy

from sklearn import model_selection


features = car_train.columns.values[1:]
print(features)

X =car_train[features]
y = car_train['safe']

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=55)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

['S1' 'S2' 'S3' 'S4' 'S5' 'S6' 'S7' 'S8' 'S9' 'S10' 'S11' 'S12' 'S13'
 'S14' 'S15' 'S16' 'S17' 'S18' 'S19' 'S20' 'S21' 'S22']
(26591, 22)
(6648, 22)
(26591,)
(6648,)


In [ ]:
# Building Decision Tree model on training data

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix


D_tree = DecisionTreeClassifier(max_depth=7)
D_tree.fit(X_train, y_train)

# Accuracy on training data
tree_predict1 = D_tree.predict(X_train)
cm1 = confusion_matrix(y_train, tree_predict1)
accuracy_train = (cm1[0,0]+cm1[1,1]) / sum(sum(cm1))
print("Training Accuracy of Decision Tree: ", accuracy_train)

# Accuracy on test data
tree_predict2 = D_tree.predict(X_test)
cm2 = confusion_matrix(y_test, tree_predict2)
accuracy_test = (cm2[0,0]+cm2[1,1]) / sum(sum(cm2))
print("Test Accuracy of Decision Tree: ", accuracy_test)

# AUC on train data



Training Accuracy of Decision Tree:  0.8814636531157158
Test Accuracy of Decision Tree:  0.8786101083032491


In [27]:
# Decision tree accuracy on train and test data 88% and 87% respectively

# Use Random Forest now

from sklearn.ensemble import RandomForestClassifier

R_forest = RandomForestClassifier(n_estimators=300, max_depth=10, max_features=4)
R_forest.fit(X_train, y_train)

# Accuracy on training data
forest_predict1 = R_forest.predict(X_train)
cm3 = confusion_matrix(y_train, forest_predict1)
accuracy_forest_train = (cm3[0,0]+cm3[1,1]) / sum(sum(cm3))
print("Training Accuracy of Random Forest: ", accuracy_forest_train)

# Accuracy on test data
forest_predict2 = R_forest.predict(X_test)
cm4 = confusion_matrix(y_test, forest_predict2)
accuracy_forest_test = (cm4[0,0]+cm4[1,1]) / sum(sum(cm4))
print("Test Accuracy of Random Forest: ", accuracy_forest_test)

# Accuracy of Random forest on train and test data 91% and 90% respectively vs 88% and 87% for decision tree
# So Random forest performs better than decision tree

Training Accuracy of Random Forest:  0.9175284870820954
Test Accuracy of Random Forest:  0.9073405535499398
